# Named TypeScript configuration: new-build verification

This notebook tests the new named Deno/TypeScript path directly, rather than changing a standalone kernel's working directory.

Configuration: **`ts-config-probe`**. Its project root is the local-only fixture printed in the outputs below. The notebook default remains unchanged; each probe cell explicitly selects this configuration.

The fixture supplies a project import alias, JSON data and custom JSX settings. It has no third-party dependencies. Its hashes are checked before and after runtime startup. The earlier lab remains open in another app instance, so this notebook avoids its writer lease.

## 1. Save, select and execute

The named profile was saved through the new Deno schema using a create-only identity guard. Saving did not start a runtime. The next cell explicitly selects `ts-config-probe`, imports its alias and JSON, then imports the TSX entrypoint to check that project JSX settings are active.

## New-build result

**Named TS/Deno configuration is now present in the live interface.** A create-only save produced profile `ts-config-probe` with stable profile ID `64e074b9-8f19-4782-b7ad-ef895c3e131f`. The probe cell below durably selects that profile through `metadata.spur.kernel_profile`. The notebook default remains the standalone factory.

**Agent/MCP execution is blocked before the cell runs.** Both the original workspace endpoint and a recovered direct app-owned endpoint returned:

```text
kernel ensure failed: unknown, foreign or superseded native slot;
inspect notebook_kernel_profiles
```

Profile discovery returns the exact saved slot and profile ID, but the cell still has no execution count or output. This is a startup/routing failure, not an alias or JSX test failure. The configuration's Save and restart request lost its transport before returning. After reconnecting, the durable profile is still saved; the recovered runtime registry reports no applied process. Restart completion is not claimed.

The first project location was rejected because an ancestor package.json could affect discovery. The isolated fixture under `/private/tmp` removes that diagnostic without changing home/project configuration. Its files are hashed before and after the probe.

The following import test remains ready to rerun after the MCP startup path is repaired. Human Run-button execution has not been tested in this check.


In [ ]:
import { configuredValue } from '@config-probe';
import probeData from './data.json' with { type: 'json' };
import { result as configuredJSX } from './entry.tsx';
const namedConfigResult = {
  profile: 'ts-config-probe',
  alias: configuredValue,
  json: probeData.message,
  jsx: configuredJSX,
  cwd: Deno.cwd(),
  runtime: Deno.version,
  checks: {
    alias: configuredValue === 'local-alias-ok',
    json: probeData.message === 'json-import-ok',
    jsx: configuredJSX.type === 'article' && configuredJSX.props.children === 'local-alias-ok'
  }
};
if (!Object.values(namedConfigResult.checks).every(Boolean)) throw new Error('Named configuration probe mismatch');
console.log(JSON.stringify(namedConfigResult,null,2));

In [2]:
// Standalone control only. This does not test the named profile or its aliases.
const standaloneModule = await import('file:///private/tmp/spur-ts-named-probe-3f23vd3j/module.ts');
console.log(JSON.stringify({probe:'standalone direct-file control',value:standaloneModule.configuredValue,pass:standaloneModule.configuredValue==='local-alias-ok',cwd:Deno.cwd(),runtime:Deno.version,executable:Deno.execPath()},null,2));

{
  "probe": "standalone direct-file control",
  "value": "local-alias-ok",
  "pass": true,
  "cwd": "/Users/kevintruong/project/spur-notebook",
  "runtime": {
    "deno": "2.8.1",
    "v8": "14.9.207.2-rusty",
    "typescript": "6.0.3"
  },
  "executable": "/Users/kevintruong/Applications/SpurLab.app/Contents/MacOS/deno"
}


## Control result and handoff

The standalone control above succeeded with bundled Deno 2.8.1 and imported the task-owned typed module by exact file URL. This confirms that general Deno execution and the local source file are usable in this new build. It does not count as a named-profile alias test.

| Step | Result |
|---|---|
| Discover Deno project configuration schema | Confirmed in the live new build |
| Preview without saving | No profile identity allocated; local dependency diagnostics retained |
| Save named profile | Confirmed with stable UUID and schema v3 |
| Select named profile on TS cell | Confirmed in saved cell metadata |
| Execute the named profile through MCP | Blocked before execution by native-slot ensure error |
| Save and restart | Connection closed before result; recovered registry is saved/unapplied |
| Standalone direct-file control | Passed |
| Project file preservation | Before/after hash check recorded in the evidence package |

The next fix should target the named Deno agent/MCP startup path. Do not treat the earlier standalone observations as proof of this new feature's absence, or this saved configuration as proof that its import test ran. Human Run-button execution, third-party prepared dependencies and ports API v2 remain untested here.

The task-owned profile is retained for reproducing the issue. Its temporary project path must exist when rerunning it. Evidence: `/Users/kevintruong/project/SpurCode/docs/research/artifacts/2026-11-18-ts-named-new-build/`.
